In [3]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
import scipy.io as readmat
import math
import os
import pandas as pd
from types import SimpleNamespace

# Get the current working directory
dir_file = os.getcwd()
test_case = '300'
deg_to_rad = math.pi / 180
# file = "excel_outputs 1/pglib_opf_case3_lmbd.xlsx"
# Load MATPOWER test case in .mat format
matpower_mat_file = readmat.loadmat(dir_file + '/power_system_test_cases/case' + test_case + '.mat',
                                    struct_as_record=False,
                                    squeeze_me=True)
# test_case = matpower_mat_file['matpower_testcase']
# # # Read Excel sheets (adjust sheet names as necessary)
# # bus = pd.read_excel(file, sheet_name='bus')
# # branch = pd.read_excel(file, sheet_name='branch')  # branch data is your "line" data
# # gen = pd.read_excel(file, sheet_name='gen')
# # gen_cost = pd.read_excel(file, sheet_name='gencost')

# # # Create an object with attributes corresponding to each data element
# # test_case = SimpleNamespace(bus=bus, branch=branch, gen=gen, gencost=gen_cost)

# # Load bus, generation, and line data
# bus = test_case.bus
# line = test_case.branch
# gen = test_case.gen
# gen_cost = test_case.gencost

# # Initialize parameters
# nEdges = len(line)  # Number of edges
# nNodes = len(bus)  # Number of nodes
# nGen = len(gen)  # Number of generators
# Pbase = 100  # MVA base
# Inf_transfer_Pmax = 10e6
# Pmax_line = 10e6 / Pbase  # Convert to p.u.
# Gmax = max(gen[:, 8]) / Pbase  # p.u.

# # Create Gurobi model
# model = gp.Model("Economic_Dispatch")

# # Decision variables
# bus_gen_ss = model.addVars(nNodes, lb=0, ub=Gmax, vtype=GRB.CONTINUOUS, name="bus_gen_ss")
# Pij_ss = model.addVars(nEdges, lb=-Pmax_line, ub=Pmax_line, vtype=GRB.CONTINUOUS, name="Pij_ss")
# theta_ss = model.addVars(nNodes, lb=-2 * math.pi, ub=2 * math.pi, vtype=GRB.CONTINUOUS, name="theta_ss")

# # Power balance constraints
# for bus_num_idx in range(nNodes):
#     bus_num = bus[bus_num_idx, 0]
#     gens = np.where(gen[:, 0] == bus_num)[0].tolist()
#     to_bus_list = np.where(line[:, 1] == bus_num)[0].tolist()
#     from_bus_list = np.where(line[:, 0] == bus_num)[0].tolist()

#     model.addConstr(
#         gp.quicksum(bus_gen_ss[np.where(bus[:, 0] == gen[gen_num, 0])[0][0]] for gen_num in gens) +
#         gp.quicksum(Pij_ss[to_bus] for to_bus in to_bus_list) -
#         gp.quicksum(Pij_ss[from_bus] for from_bus in from_bus_list) ==
#         (bus[bus_num_idx, 2] / Pbase) + (bus[bus_num_idx, 4] / Pbase),
#         name=f"power_balance_{bus_num}"
#     )

# # Generator power limit constraints
# for gen_num in range(nGen):
#     bus_idx = np.where(bus[:, 0] == gen[gen_num, 0])[0][0]
#     model.addConstr(bus_gen_ss[bus_idx] <= gen[gen_num, 8] / Pbase, name=f"gen_max_{gen_num}")
#     model.addConstr(bus_gen_ss[bus_idx] >= gen[gen_num, 9] / Pbase, name=f"gen_min_{gen_num}")

# # Non-generating buses should not generate power
# for bus_num_idx in range(nNodes):
#     bus_num = bus[bus_num_idx, 0]
#     if not np.any(np.equal(gen[:, 0], bus_num)):
#         model.addConstr(bus_gen_ss[bus_num_idx] == 0, name=f"no_gen_{bus_num_idx}")

# # Active power flow constraints on each line
# for line_num in range(nEdges):
#     if line[line_num, 8] == 0:
#         line[line_num, 8] = 1  # Set tap ratio to 1 for transmission lines

#     reciprocal_term = 1 / (line[line_num, 3] * line[line_num, 8])
#     from_bus = np.where(bus[:, 0] == line[line_num, 0])[0][0]
#     to_bus = np.where(bus[:, 0] == line[line_num, 1])[0][0]

#     model.addConstr(
#         Pij_ss[line_num] == reciprocal_term * (theta_ss[from_bus] - theta_ss[to_bus] - (line[line_num, 9] * deg_to_rad)),
#         name=f"power_flow_{line_num}"
#     )

# # Thermal limits on lines
# for line_num in range(nEdges):
#     if line[line_num, 5] == 0:
#         line[line_num, 5] = Pmax_line  # Set default line limit

#     model.addConstr(Pij_ss[line_num] <= line[line_num, 5] / Pbase, name=f"line_max_{line_num}")
#     model.addConstr(Pij_ss[line_num] >= -line[line_num, 5] / Pbase, name=f"line_min_{line_num}")

# # Angle difference constraints
# for angle_num in range(nEdges):
#     from_bus = np.where(bus[:, 0] == line[angle_num, 0])[0][0]
#     to_bus = np.where(bus[:, 0] == line[angle_num, 1])[0][0]

#     model.addConstr(
#         (theta_ss[from_bus] - theta_ss[to_bus]) <= line[angle_num, 12] * deg_to_rad,
#         name=f"angle_max_{angle_num}"
#     )
#     model.addConstr(
#         (theta_ss[from_bus] - theta_ss[to_bus]) >= line[angle_num, 11] * deg_to_rad,
#         name=f"angle_min_{angle_num}"
#     )

# # Identifying slack bus and setting its angle to zero
# slack_bus = np.where(bus[:, 1] == 3)[0][0]
# model.addConstr(theta_ss[slack_bus] == 0, name="slack_bus")

# # Objective function: Minimize generation cost
# expr = gp.quicksum(
#     gen_cost[gen_num, 4] * (bus_gen_ss[np.where(bus[:, 0] == gen[gen_num, 0])[0][0]] * Pbase) +
#     gen_cost[gen_num, 5] * (bus_gen_ss[np.where(bus[:, 0] == gen[gen_num, 0])[0][0]] * Pbase) +
#     gen_cost[gen_num, 6]
#     for gen_num in range(nGen)
# )

# model.setObjective(expr, GRB.MINIMIZE)
# # Disable presolving
# model.setParam("Presolve", 0)

# # Disable primal heuristics
# model.setParam("Heuristics", 0)

# # Solve the model
# model.optimize()

# # Display results
# if model.status == GRB.OPTIMAL:
#     print("\nOptimal Generation:")
#     for k in range(nGen):
#         bus_idx = np.where(bus[:, 0] == gen[k, 0])[0][0]
#         print(f"G[{gen[k, 0]}] = {bus_gen_ss[bus_idx].X:.6f}")

#     print("\nPower Flow:")
#     for k in range(nEdges):
#         print(f"Pij[{k + 1}] = {Pij_ss[k].X:.6f}")

#     print("\nBus Angles:")
#     for k in range(nNodes):
#         print(f"theta[{k + 1}] = {theta_ss[k].X * 1 / deg_to_rad:.6f} degrees")
# else:
#     print("Optimization did not reach an optimal solution.")

In [7]:
# Retrieve basis statuses for the decision variables and constraints
var_basis = model.getAttr(GRB.Attr.VBasis, model.getVars())
constr_basis = model.getAttr(GRB.Attr.CBasis, model.getConstrs())
# Print basis statuses for each variable
print("Variable basis statuses:")
for var, status in zip(model.getVars(), var_basis):
    # Basis status codes typically are:
    # 0: basic, -1: nonbasic at lower bound, -2: nonbasic at upper bound, -3: nonbasic free
    print(f"  {var.VarName}: {status}")
# Print basis statuses for each constraint
print("\nConstraint basis statuses:")
for constr, status in zip(model.getConstrs(), constr_basis):
    print(f"  {constr.ConstrName}: {status}")

Variable basis statuses:
  bus_gen_ss[0]: 0
  bus_gen_ss[1]: 0
  bus_gen_ss[2]: 0
  bus_gen_ss[3]: 0
  bus_gen_ss[4]: 0
  bus_gen_ss[5]: 0
  bus_gen_ss[6]: 0
  bus_gen_ss[7]: -1
  bus_gen_ss[8]: 0
  bus_gen_ss[9]: -1
  bus_gen_ss[10]: 0
  bus_gen_ss[11]: 0
  bus_gen_ss[12]: 0
  bus_gen_ss[13]: 0
  bus_gen_ss[14]: 0
  bus_gen_ss[15]: 0
  bus_gen_ss[16]: 0
  bus_gen_ss[17]: 0
  bus_gen_ss[18]: -1
  bus_gen_ss[19]: 0
  bus_gen_ss[20]: 0
  bus_gen_ss[21]: 0
  bus_gen_ss[22]: 0
  bus_gen_ss[23]: 0
  bus_gen_ss[24]: 0
  bus_gen_ss[25]: 0
  bus_gen_ss[26]: 0
  bus_gen_ss[27]: 0
  bus_gen_ss[28]: 0
  bus_gen_ss[29]: 0
  bus_gen_ss[30]: 0
  bus_gen_ss[31]: 0
  bus_gen_ss[32]: 0
  bus_gen_ss[33]: 0
  bus_gen_ss[34]: 0
  bus_gen_ss[35]: 0
  bus_gen_ss[36]: 0
  bus_gen_ss[37]: 0
  bus_gen_ss[38]: 0
  bus_gen_ss[39]: 0
  bus_gen_ss[40]: 0
  bus_gen_ss[41]: 0
  bus_gen_ss[42]: 0
  bus_gen_ss[43]: 0
  bus_gen_ss[44]: 0
  bus_gen_ss[45]: 0
  bus_gen_ss[46]: 0
  bus_gen_ss[47]: 0
  bus_gen_ss[48]: 0
  

In [1]:
# import pandas as pd
# import json
# import glob
# import os

# # Folder containing the Excel files
# folder_path = 'excel_outputs 1'

# # Find all Excel files in the folder
# excel_files = glob.glob(os.path.join(folder_path, '*.xlsx'))

# for file in excel_files:
#     # Read all sheets from the Excel file into a dictionary
#     # The keys are sheet names and the values are DataFrames
#     excel_data = pd.read_excel(file, sheet_name=None)
    
# #     Convert each DataFrame to a list of dictionaries (one per row)
#     data_dict = {sheet: df.to_dict(orient='records') for sheet, df in excel_data.items()}
    
#     # Construct the output file name by replacing .xlsx with .json
#     base_name = os.path.splitext(os.path.basename(file))[0]
#     output_file = os.path.join(folder_path, base_name + '.json')
    
#     # Write the dictionary to a JSON file with pretty printing
#     with open(output_file, 'w', encoding='utf-8') as f:
#         json.dump(data_dict, f, indent=4)
    
#     print(f'Saved {output_file}')